# セッションステートと Agent コールバックの利用

このノートブックでは、before_agent コールバックと after_agent コールバックを利用する例を紹介します。

ユーザーの質問回数をステートに保存して、質問回数を3回までに制限します。

## 事前準備

**[SAC-01]**

ADK と Gemini API の利用に必要なパッケージをインストールします。

ADK は開発の速度が速く、後方互換性のない機能変更が行われることがあります。ここでは、安全のために動作確認ができたバーションを指定してインストールしています。

**最後に `RESTART SESSION` ボタンが表示された場合は、これをクリックしてセッションを再起動してください。**

**注意**: `ERROR: pip's dependency resolver does not currently take into account...` のようなエラーメッセージが表示される場合がありますが、これは無視して構いません。

In [ ]:
%pip install --upgrade --user \
    google-adk==2.8.0 \
    google-cloud-aiplatform==2.0.1 \
    google-genai==2.20.0

**[SAC-02]**

インストールされたパッケージのバージョンを確認します。

In [1]:
!pip list | grep -E "(google-adk|google-genai|google-cloud-aiplatform)"

google-adk                            2.8.0
google-cloud-aiplatform               2.0.1
google-genai                          2.20.0


下記の内容が出力されたことを確認してください。

```
google-adk                               2.8.0
google-cloud-aiplatform                  2.0.1
google-genai                             2.20.0
```

## ユーザー認証

**[SAC-03]**

変数 `PROJECT_ID` に事前に準備したプロジェクトのプロジェクト ID を指定してください。

**注意**: プロジェクトを作成したユーザーアカウントと Colab を使用中のユーザーアカウントが一致している必要があります。


In [ ]:
###
PROJECT_ID = 'Project ID を入力'
###

if (not PROJECT_ID) or PROJECT_ID == 'Project ID を入力':
    print('Gemini API を使用する Google Cloud Project の Project ID を入力してください。')
else:
    print(f'変数 PROJECT_ID を設定しました。')
    print(f'PROJECT_ID = {PROJECT_ID}')

Gemini API を使用する Google Cloud Project の Project ID を入力してください。


**[SAC-04]**

Google Cloud のプロジェクトを使用するためのユーザー認証を行います。

ポップアップ画面の指示に従って、認証処理を行ってください。

許可する操作を選択するチェックボックスが表示された場合は、すべてにチェックを入れてください。

In [3]:
from google.colab import auth
auth.authenticate_user(project_id=PROJECT_ID)

## 初期設定

**[SAC-05]**

この後の作業に必要なモジュールをインポートして、LlmAgent オブジェクトが参照する環境変数を設定します。

In [4]:
import os
from IPython.display import HTML, Markdown, display
import agentplatform
from agentplatform.frameworks import AdkApp
from google.adk.agents.llm_agent import LlmAgent
from google.genai.types import Content, Part

agentplatform.init(project=PROJECT_ID, location='us-central1')
os.environ['GOOGLE_CLOUD_PROJECT'] = PROJECT_ID
os.environ['GOOGLE_CLOUD_LOCATION'] = 'global'
os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'True'

**[SAC-06]**

AdkApp オブジェクトとの会話を行う簡易的なアプリケーションのクラス ChatClient を定義します。

In [5]:
class ChatClient:
    def __init__(self, adk_app, user_id='default_user'):
        self.adk_app = adk_app
        self.user_id = user_id
        self.session_id = None

    async def async_stream_query(self, message):
        if not self.session_id:
            session = await self.adk_app.async_create_session(
                user_id=self.user_id,
            )
            self.session_id = session['id']

        result = []
        async for event in self.adk_app.async_stream_query(
            user_id=self.user_id,
            session_id=self.session_id,
            message=message,
        ):
            if ('content' in event and 'parts' in event['content']):
                response = '\n'.join(
                    [p['text'] for p in event['content']['parts'] if 'text' in p]
                )
                if response:
                    result.append(response)
        return '\n'.join(result)

## Agent コールバックの定義

**[SAC-07]**

before_agent コールバックで使用するコールバック関数を定義します。

In [6]:
async def count_questions_callback(callback_context):

    agent_name = callback_context.agent_name
    invocation_id = callback_context.invocation_id
    print(f'\n[Before Agent Callback] {invocation_id}')

    request_count = callback_context.state.get('request_count', 0)
    if request_count == 3:
        return Content(
            parts=[Part.from_text(text=f'{agent_name}への質問は3回までです。')],
            role='model',
        )

    request_count += 1
    callback_context.state['request_count'] = request_count
    return None

**[SAC-08]**

after_agent コールバックで使用するコールバック関数を定義します。

In [7]:
async def add_message_callback(callback_context):

    agent_name = callback_context.agent_name
    invocation_id = callback_context.invocation_id
    print(f'\n[After Agent Callback] {invocation_id}')

    request_count = callback_context.state['request_count']
    return Content(
        parts=[Part.from_text(text=f'\n{agent_name}が{request_count}回目の質問に回答しました。')],
        role='model',
    )

## LlmAgent オブジェクトと AdkApp オブジェクトの作成

**[SAC-09]**

ユーザーの質問に回答する AI エージェントを作成します。

コールバック関数を利用して、質問回数をカウントした上で、質問回数を3回までに制限しています。

In [8]:
instruction = '''
あなたはユーザーの質問に回答するエージェントです。
- はじめに "{request_count}回目の質問ですね！" と言ってから回答してください。
- その後は、回答だけを簡潔な1文で返してください。
'''

qa_agent = LlmAgent(
    name='qa_agent',
    model='gemini-3.5-flash-lite',
    description='質問に回答するエージェント',
    instruction=instruction,
    before_agent_callback=count_questions_callback,
    after_agent_callback=add_message_callback,
)

qa_agent_app = AdkApp(
    agent=qa_agent,
    app_name='qa_agent_app',
)

**[SAC-10]**

1回目の質問をします。

In [9]:
chat_client = ChatClient(qa_agent_app)

query = '''
フィンランドの首都は？
'''
response = await chat_client.async_stream_query(query)
print('====')
display(Markdown(response))

/root/.local/lib/python3.13/site-packages/agentplatform/frameworks/adk.py:1146: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/root/.local/lib/python3.13/site-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()



[Before Agent Callback] e-c184413e-8099-40ed-85ca-063070157b7e



[After Agent Callback] e-c184413e-8099-40ed-85ca-063070157b7e
====


1回目の質問ですね！
フィンランドの首都はヘルシンキです。

qa_agentが1回目の質問に回答しました。

**[SAC-11]**

2回目の質問をします。

In [10]:
query = '''
ノルウェーの首都は？
'''
response = await chat_client.async_stream_query(query)
print('====')
display(Markdown(response))


[Before Agent Callback] e-99341914-81db-4340-a056-7f0fa8c03b9a

[After Agent Callback] e-99341914-81db-4340-a056-7f0fa8c03b9a
====


2回目の質問ですね！
ノルウェーの首都はオスロです。

qa_agentが2回目の質問に回答しました。

**[SAC-12]**

3回目の質問をします。

In [11]:
query = '''
スペインの首都は？
'''
response = await chat_client.async_stream_query(query)
print('====')
display(Markdown(response))


[Before Agent Callback] e-268107c8-ebe6-4c4a-a724-6a918958142a

[After Agent Callback] e-268107c8-ebe6-4c4a-a724-6a918958142a
====


3回目の質問ですね！
スペインの首都はマドリードです。

qa_agentが3回目の質問に回答しました。

**[SAC-13]**

4回目の質問をします。

In [12]:
query = '''
日本の首都は？
'''
response = await chat_client.async_stream_query(query)
print('====')
display(Markdown(response))


[Before Agent Callback] e-216ee990-93b3-4db2-af26-0ecd633d4135
====


qa_agentへの質問は3回までです。